In [ ]:
#| default_exp parser

# Notebook text and Python parsing

> Normalize notebook text, directives and Python syntax without a notebook model.

Parsing owns source conversion and AST inspection. The data model imports these operations so plain text and syntax errors have one representation throughout the package.

In [ ]:
#| export
import ast, re
from fastcore.nbio import NbCell, mk_cell
from nbskill.diagnostics import api_error, source_without_directives

#| export
## Turning text into notebook cells

#| export
Write tools accept plain text because that is what agents and humans can produce quickly. This section turns that text into real cells: split on block separators, honor `%%markdown`/`%%code` markers, and split exported code into symbol-sized chunks when needed. It is the small parser that makes notebook edits feel like editing a document instead of assembling JSON.

In [ ]:
#| exporti
#| hide
def _split_blocks(text):
    text = "" if text is None else str(text)
    if not text: return []
    return [o.strip("\n") for o in re.split(r"(?m)^\s*---\s*$", text) if o.strip()]

In [ ]:
#| exporti
#| hide
def _coerce_cell(cell, default_type="code"):
    if isinstance(cell, dict): return cell
    if isinstance(cell, (tuple, list)) and len(cell) == 2:
        cell_type, source = cell
        return mk_cell(str(source), cell_type=str(cell_type))
    return mk_cell(str(cell), cell_type=default_type)

`symbol_short_name` derives a display alias from a qualified name. Keep the qualified name in records because different modules can expose the same short name.

`symbol_short_name` removes the owner prefix when a caller needs only the local symbol name.

In [ ]:
#| export
def symbol_short_name(symbol):
    "Return the final dotted component of a symbol name."
    return str(symbol).rsplit(".", 1)[-1]

In [ ]:
assert symbol_short_name("Cell.source") == "source"

In [ ]:
#| exporti
#| hide
def _name_parts(node):
    if isinstance(node, ast.Name): return [node.id]
    if isinstance(node, ast.Attribute):
        base = _name_parts(node.value)
        return [*base, node.attr] if base else [node.attr]
    return []

Graph records retain dotted names through `call_name`. `symbol_short_name` and `short_call_name` derive display aliases from those names; a call target that cannot be named remains `None` or the supplied default.

`call_name` preserves qualification when resolving a call target.

In [ ]:
#| export
def call_name(node):
    "Return dotted name text for an AST name, attribute, or call expression."
    parts = _name_parts(node.func if isinstance(node, ast.Call) else node)
    return ".".join(parts) if parts else None

In [ ]:
qualified_call = ast.parse("package.calculate(value)").body[0].value
assert call_name(qualified_call) == "package.calculate"

In [ ]:
#| export
def call_sites(node, source=""):
    "Return dotted call names and source locations from one parsed Python tree."
    lines, records = str(source or "").splitlines(), []
    for child in ast.walk(node):
        if not isinstance(child, ast.Call) or not (name := call_name(child.func)): continue
        lineno = getattr(child, "lineno", None)
        line = lines[lineno-1].strip() if lineno and lineno <= len(lines) else ""
        records.append(dict(name=name, lineno=lineno, line=line))
    return tuple(records)

call_sites gives calls one representation in foundation. Graph navigation can add short aliases while retaining the original source locations.

In [ ]:
#| exportd
calls_source = "result = package.calculate(value)"
assert call_sites(ast.parse(calls_source), calls_source) == (dict(name="package.calculate", lineno=1, line=calls_source),)
call_sites(ast.parse(calls_source), calls_source)

`short_call_name` selects the local component of an AST call target.

In [ ]:
#| export
def short_call_name(node, default=None):
    'Return the final dotted component of a call/name node, or `default`.'
    name = call_name(node)
    return default if name is None else symbol_short_name(name)

In [ ]:
assert short_call_name(ast.parse("package.calculate(value)").body[0].value) == "calculate"

`is_definition_node` recognizes classes as well as synchronous and asynchronous functions. Definition consumers share this predicate instead of maintaining different AST type lists.

`is_definition_node` distinguishes declarations from ordinary statements.

In [ ]:
#| export
def is_definition_node(node):
    'Return whether an AST node defines a function or class.'
    return isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))

In [ ]:
assert is_definition_node(ast.parse("def calculate(): return 42").body[0])
assert not is_definition_node(ast.parse("answer = 42").body[0])

`node_start_line` includes decorators when selecting a definition for editing.

In [ ]:
#| export
def node_start_line(node):
    'Return the zero-based start line including decorators.'
    return min([node.lineno, *[d.lineno for d in getattr(node, "decorator_list", [])]]) - 1

In [ ]:
decorated_definition = ast.parse("@cache\ndef calculate(): return 42").body[0]
assert node_start_line(decorated_definition) == 0

The shared directive reader uses fastcore to merge comment and metadata directives. This keeps export detection and execution consistent with nbdev 3.3, including colon variants and comment precedence.

In [ ]:
#| export
def cell_directives(cell) -> dict:
    "Return normalized nbdev directives, with comments taking precedence over metadata."
    return NbCell(None, cell).directives

cell_directives merges metadata and leading comments through fastcore. Using one parser prevents export detection from disagreeing with execution about the same cell.

In [ ]:
normalized_directives = cell_directives(mk_cell("#| export: true\n#| eval false\nvalue = 42"))
normalized_directives

In [ ]:
#| hide
assert cell_directives(mk_cell("#| export: true\n#| eval false")) == {"export": "", "eval": "false"}

In [ ]:
#| export
def is_export_directive(line):
    "Return whether a line directs executable code to a module."
    return bool({"export", "exports", "exporti"} & cell_directives(mk_cell(line)).keys())

is_export_directive recognizes executable export directives through the shared parser, including their colon spellings.

In [ ]:
is_export_directive("#| export: true")

In [ ]:
#| hide
assert is_export_directive("#| exporti: true")
assert not is_export_directive("#| exportd")

In [ ]:
#| exporti
#| hide
def _is_export_gap(lines): return bool(lines) and all((not line.strip()) or is_export_directive(line) for line in lines)

In [ ]:
#| exporti
#| hide
def _emit_code_chunk(chunks, lines, export_prefix=None):
    if export_prefix: lines = [*export_prefix, *lines]
    text = "\n".join(lines).strip("\n")
    if text: chunks.append(text)

Python notebook blocks can contain exported and non-exported regions. The splitter retains source order and separates those regions before creating cells.

In [ ]:
#| exporti
#| hide
def _split_code_cell_sources(source):
    source = source.strip("\n")
    if not source: return []
    try: tree = ast.parse(source)
    except SyntaxError: return [source]
    if sum(1 for node in tree.body if is_definition_node(node)) <= 1: return [source]

    lines = source.splitlines()
    first_start = node_start_line(tree.body[0]) if tree.body else 0
    leading = lines[:first_start]
    shared_export = [line for line in leading if is_export_directive(line)] if _is_export_gap(leading) else []
    chunks = []
    if shared_export: cursor = first_start
    else:
        _emit_code_chunk(chunks, leading)
        cursor = first_start

    for node in tree.body:
        start = node_start_line(node)
        end = node.end_lineno
        gap = lines[cursor:start]
        if shared_export and _is_export_gap(gap): gap = []
        _emit_code_chunk(chunks, [*gap, *lines[start:end]], shared_export or None)
        cursor = end
    _emit_code_chunk(chunks, lines[cursor:])
    return chunks or [source]

In [ ]:
#| exporti
#| hide
def _split_code_cell(cell):
    cell_type = cell.get("cell_type") if isinstance(cell, dict) else getattr(cell, "cell_type", None)
    if cell_type != "code": return [cell]
    source = cell.get("source", "") if isinstance(cell, dict) else getattr(cell, "source", "")
    if isinstance(source, list): source = "".join(source)
    sources = _split_code_cell_sources(str(source))
    if len(sources) <= 1: return [cell]
    return [mk_cell(source, cell_type="code") for source in sources]

Multiple definitions in an exported cell become separate cells. Their order and export directives remain intact.

In [ ]:
#| exporti
#| hide
def _split_symbol_cells(cells):
    split = []
    for cell in cells: split.extend(_split_code_cell(cell))
    return split

In [ ]:
#| exporti
#| hide
def _cell_from_block(block, default_type="code"):
    lines = block.splitlines()
    marker = lines[0].strip().lower() if lines else ""
    cell_type = default_type
    if marker in {"%%markdown", "%%md"}: cell_type, lines = "markdown", lines[1:]
    elif marker == "%%code": cell_type, lines = "code", lines[1:]
    elif marker == "%%raw": cell_type, lines = "raw", lines[1:]
    return mk_cell("\n".join(lines), cell_type=cell_type)


`parse_cells` normalizes text blocks or existing cell structures before splitting exported definitions into separate cells.

In [ ]:
#| export
def parse_cells(cells, default_type="code"):
    'Normalize cell structures or marked text blocks into notebook cells.'
    if isinstance(cells, (list, tuple)): return _split_symbol_cells([_coerce_cell(o, default_type) for o in cells])

    parsed = [_cell_from_block(block, default_type) for block in _split_blocks(cells)]
    return _split_symbol_cells(parsed)

In [ ]:
parsed_cells = parse_cells("%%markdown\n## Results\n---\n%%code\nanswer = 42")
assert [cell.cell_type for cell in parsed_cells] == ["markdown", "code"]
[cell.source for cell in parsed_cells]

`parse_one_cell` enforces the single-cell boundary used by replacement operations. Multiple blocks need a structural edit instead.

In [ ]:
#| export
def parse_one_cell(text, default_type="code"):
    "Normalize one cell block and reject multi-cell replacements."
    if isinstance(text, (list, tuple)):
        if len(text) != 1: api_error("parse_one_cell requires one cell; use edit_notebook with structural operations for multiple cells")
        return _coerce_cell(text[0], default_type)
    blocks = _split_blocks(text)
    if len(blocks) != 1: api_error("parse_one_cell requires one block; use parse_cells or edit_notebook for multiple cells")
    return _cell_from_block(blocks[0], default_type)

In [ ]:
single_cell = parse_one_cell("%%code\nanswer = 42")
assert single_cell.source == "answer = 42"
single_cell.source

In [ ]:
mixed_cells = _split_symbol_cells([mk_cell("#| export\ndef first(): return 1\n\ndef second(): return 2")])
assert len(mixed_cells) == 2
assert "def first" in mixed_cells[0].source and "def second" in mixed_cells[1].source
assert all(cell.source.startswith("#| export") for cell in mixed_cells)
[cell.source for cell in mixed_cells]

The notebook format permits source as one string or a list of fragments. `cell_source` normalizes both at the read boundary; the cell object remains the source of metadata and outputs.

In [ ]:
#| export
def cell_source(cell):
    'Return cell source as one string for mapping and object inputs.'
    source = cell.get("source", "") if isinstance(cell, dict) else getattr(cell, "source", "")
    if isinstance(source, list): return "".join(source)
    return str(source)

In [ ]:
assert cell_source(dict(cell_type="code", source=["answer = ", "42"])) == "answer = 42"

`parse_cell` retains the syntax error alongside the AST result. Source analysis needs that distinction to report malformed Python without dropping the original cell.

In [ ]:
#| export
def parse_cell(cell):
    "Return a Python AST and syntax error separately without changing the cell."
    if cell.get("cell_type") != "code": return None, None
    source = source_without_directives(cell_source(cell))
    if re.match(r"^\s*%%prompt(?:\s|$)", source): return None, None
    try: return ast.parse(source), None
    except SyntaxError as error: return None, error

In [ ]:
parsed_example, parse_error = parse_cell(mk_cell("answer = 42"))
assert isinstance(parsed_example, ast.Module) and parse_error is None
ast.dump(parsed_example)

In [ ]:
#| hide
assert isinstance(parse_cell(mk_cell("def broken("))[1], SyntaxError)

`parse_code_cell` returns a tree for valid code and `None` for malformed or non-code cells. Use `cell_analysis` when the reason for a failed parse matters.

In [ ]:
#| export
def parse_code_cell(cell):
    "Return the Python tree, or None for non-code and unparseable source."
    return parse_cell(cell)[0]

In [ ]:
parsed = parse_code_cell(mk_cell("answer = 42"))
assert isinstance(parsed.body[0], ast.Assign)
assert parse_code_cell(mk_cell("answer =")) is None
assert parse_code_cell(mk_cell("# Heading", cell_type="markdown")) is None
parsed.body[0].targets[0].id

In [ ]:
#| hide
assert parse_code_cell(mk_cell("def broken(")) is None

`parse_code_cell` keeps its tree-or-None interface for AST consumers. Use `cell_analysis` when a caller must distinguish a parse failure from non-code and inspect the original exception.